In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
random_seed = 42

In [3]:
ALL_COLUMNS = [
    "sample_id",
    "device",
    "active_fault",
    "speed_rpm",
    "flow_m3h",
    "differential_head_m",
    "suction_abs_pressure_bar",
    "discharge_pressure_bar",
    "ambient_temp_c",
    "fluid_temp_c",
    "fluid_density_kgm3",
    "efficiency",
    "hydraulic_power_kw",
    "shaft_power_kw",
    "motor_power_kw",
    "motor_current_a",
    "vibration_mm_s",
    "bearing_temp_c",
    "npsha_m",
    "npshr_m",
    "cavitation_margin_m",
    "fault_cavitation",
    "fault_impeller_wear",
    "fault_bearing_friction",
    "is_faulted",
    "fault_labels",
    "fault_severity",
]

In [4]:
def severity_label(values):
    values = np.asarray(values, dtype=float)
    labels = np.full(len(values), "none", dtype=object)
    labels[(values > 0.0) & (values < 0.33)] = "low"
    labels[(values >= 0.33) & (values < 0.66)] = "medium"
    labels[values >= 0.66] = "high"
    return labels


def water_density_kgm3(temp_c):
    t = np.asarray(temp_c, dtype=float)
    return 1000.0 * (
        1.0
        - ((t + 288.9414) / (508929.2 * (t + 68.12963))) * (t - 3.9863) ** 2
    )


def water_vapor_pressure_pa(temp_c):
    t = np.asarray(temp_c, dtype=float)
    log10_p_mmHg = 8.07131 - 1730.63 / (233.426 + t)
    return (10.0 ** log10_p_mmHg) * 133.322368

In [5]:
def make_device_params(rng, overrides=None):
    q_bep_m3h = float(np.clip(rng.normal(90.0, 4.0), 78.0, 100.0))
    h_bep_m = float(np.clip(rng.normal(45.0, 1.8), 40.0, 50.0))
    h0_m = float(np.clip(1.24 * h_bep_m + rng.normal(0.0, 0.8), 1.18 * h_bep_m, 1.30 * h_bep_m))

    params = {
        "nominal_speed_rpm": float(np.clip(rng.normal(1480.0, 18.0), 1440.0, 1510.0)),
        "q_bep_m3h": q_bep_m3h,
        "h_bep_m": h_bep_m,
        "h0_m": h0_m,
        "efficiency_peak": float(np.clip(rng.normal(0.82, 0.015), 0.76, 0.86)),
        "npshr_bep_m": float(np.clip(rng.normal(3.0, 0.25), 2.4, 3.8)),
        "motor_efficiency": float(np.clip(rng.normal(0.93, 0.01), 0.90, 0.96)),
        "motor_power_factor": float(np.clip(rng.normal(0.86, 0.02), 0.80, 0.90)),
        "line_voltage_v": 400.0,
        "sensor_noise_scale": float(np.clip(rng.normal(1.0, 0.06), 0.88, 1.12)),
        "thermal_bias_c": float(np.clip(rng.normal(0.0, 0.8), -2.0, 2.0)),
    }

    if overrides:
        params.update(overrides)

    q_bep_m3s = params["q_bep_m3h"] / 3600.0
    params["pump_curve_k"] = (params["h0_m"] - params["h_bep_m"]) / max(q_bep_m3s ** 2, 1e-9)
    return params

In [6]:
def efficiency_curve(flow_m3s, q_bep_m3s, efficiency_peak):
    ratio = np.divide(flow_m3s, np.maximum(q_bep_m3s, 1e-9))
    eta = efficiency_peak * (1.0 - 0.26 * (ratio - 1.0) ** 2)
    return np.clip(eta, 0.35, efficiency_peak)

In [7]:
def solve_operating_point(params, speed_rpm, static_head_m, system_k):
    speed_ratio = speed_rpm / params["nominal_speed_rpm"]
    h0 = params["h0_m"] * speed_ratio ** 2
    numerator = np.maximum(h0 - static_head_m, 0.0)
    denominator = params["pump_curve_k"] + system_k
    flow_m3s = np.sqrt(np.divide(numerator, np.maximum(denominator, 1e-9)))
    head_m = static_head_m + system_k * flow_m3s ** 2
    q_bep_m3s = (params["q_bep_m3h"] / 3600.0) * speed_ratio
    efficiency = efficiency_curve(flow_m3s, q_bep_m3s, params["efficiency_peak"])
    return flow_m3s, head_m, efficiency


def sample_conditions(rng, n, fault_type):
    speed_ratio = rng.uniform(0.78, 1.04, n)
    ambient_temp_c = rng.uniform(8.0, 38.0, n)
    fluid_temp_c = ambient_temp_c + rng.uniform(1.0, 10.0, n)

    if fault_type == "cavitation":
        fluid_temp_c = np.clip(rng.uniform(24.0, 55.0, n), 5.0, 60.0)
        ambient_temp_c = np.clip(fluid_temp_c - rng.uniform(2.0, 10.0, n), 5.0, 45.0)
        suction_abs_pressure_bar = rng.uniform(1.08, 1.28, n)
        suction_static_head_m = rng.uniform(0.2, 2.0, n)
        suction_loss_k = rng.uniform(2500.0, 7000.0, n)
    else:
        suction_abs_pressure_bar = rng.uniform(1.20, 1.70, n)
        suction_static_head_m = rng.uniform(1.0, 4.0, n)
        suction_loss_k = rng.uniform(800.0, 3500.0, n)

    static_head_m = rng.uniform(10.0, 26.0, n)
    system_k = rng.uniform(9000.0, 28000.0, n)

    return {
        "speed_ratio": speed_ratio,
        "ambient_temp_c": ambient_temp_c,
        "fluid_temp_c": fluid_temp_c,
        "suction_abs_pressure_bar": suction_abs_pressure_bar,
        "suction_static_head_m": suction_static_head_m,
        "suction_loss_k": suction_loss_k,
        "static_head_m": static_head_m,
        "system_k": system_k,
    }

In [8]:
def motor_current_a(motor_power_kw, line_voltage_v, motor_power_factor):
    return (motor_power_kw * 1000.0) / (np.sqrt(3.0) * line_voltage_v * motor_power_factor)


def recompute_power_columns(df, motor_efficiency, line_voltage_v, motor_power_factor):
    hydraulic_power_kw = (
        df["fluid_density_kgm3"].to_numpy()
        * 9.81
        * (df["flow_m3h"].to_numpy() / 3600.0)
        * df["differential_head_m"].to_numpy()
        / 1000.0
    )
    shaft_power_kw = hydraulic_power_kw / np.maximum(df["efficiency"].to_numpy(), 0.35)
    motor_power_kw = shaft_power_kw / motor_efficiency

    df["hydraulic_power_kw"] = hydraulic_power_kw
    df["shaft_power_kw"] = shaft_power_kw
    df["motor_power_kw"] = motor_power_kw
    df["motor_current_a"] = motor_current_a(motor_power_kw, line_voltage_v, motor_power_factor)
    df["discharge_pressure_bar"] = df["suction_abs_pressure_bar"] + (
        df["fluid_density_kgm3"] * 9.81 * df["differential_head_m"] / 1e5
    )
    return df

In [9]:
def build_healthy_samples(device_id, params, conditions, rng, sample_offset=0):
    n = len(conditions["speed_ratio"])
    noise_scale = params["sensor_noise_scale"]

    speed_rpm = params["nominal_speed_rpm"] * conditions["speed_ratio"]
    flow_m3s, head_m, efficiency = solve_operating_point(
        params=params,
        speed_rpm=speed_rpm,
        static_head_m=conditions["static_head_m"],
        system_k=conditions["system_k"],
    )

    flow_m3h = 3600.0 * flow_m3s
    fluid_density = water_density_kgm3(conditions["fluid_temp_c"])
    hydraulic_power_kw = fluid_density * 9.81 * flow_m3s * head_m / 1000.0
    shaft_power_kw = hydraulic_power_kw / efficiency
    motor_power_kw = shaft_power_kw / params["motor_efficiency"]
    current_a = motor_current_a(
        motor_power_kw,
        params["line_voltage_v"],
        params["motor_power_factor"],
    )

    vapor_pressure_pa = water_vapor_pressure_pa(conditions["fluid_temp_c"])
    suction_pressure_pa = conditions["suction_abs_pressure_bar"] * 1e5
    suction_loss_m = conditions["suction_loss_k"] * flow_m3s ** 2
    npsha_m = (
        (suction_pressure_pa - vapor_pressure_pa) / (fluid_density * 9.81)
        + conditions["suction_static_head_m"]
        - suction_loss_m
    )

    speed_ratio = speed_rpm / params["nominal_speed_rpm"]
    q_bep_m3s = (params["q_bep_m3h"] / 3600.0) * speed_ratio
    flow_ratio = np.divide(flow_m3s, np.maximum(q_bep_m3s, 1e-9))
    npshr_m = params["npshr_bep_m"] * np.clip(speed_ratio, 0.6, None) ** 2 * np.clip(flow_ratio, 0.55, 1.35) ** 1.8
    cavitation_margin_m = npsha_m - npshr_m

    off_bep = np.abs(flow_ratio - 1.0)
    vibration_mm_s = 0.9 + 2.2 * off_bep ** 1.5 + 0.015 * shaft_power_kw
    bearing_temp_c = conditions["ambient_temp_c"] + 16.0 + 0.55 * shaft_power_kw + 5.0 * off_bep ** 1.6 + params["thermal_bias_c"]
    discharge_pressure_bar = conditions["suction_abs_pressure_bar"] + fluid_density * 9.81 * head_m / 1e5

    data = pd.DataFrame({
        "sample_id": np.arange(sample_offset, sample_offset + n),
        "device": device_id,
        "active_fault": "healthy",
        "speed_rpm": speed_rpm + rng.normal(0.0, 2.0 * noise_scale, n),
        "flow_m3h": flow_m3h + rng.normal(0.0, 0.35 * noise_scale, n),
        "differential_head_m": head_m + rng.normal(0.0, 0.12 * noise_scale, n),
        "suction_abs_pressure_bar": conditions["suction_abs_pressure_bar"] + rng.normal(0.0, 0.01 * noise_scale, n),
        "discharge_pressure_bar": discharge_pressure_bar + rng.normal(0.0, 0.015 * noise_scale, n),
        "ambient_temp_c": conditions["ambient_temp_c"] + rng.normal(0.0, 0.15 * noise_scale, n),
        "fluid_temp_c": conditions["fluid_temp_c"] + rng.normal(0.0, 0.12 * noise_scale, n),
        "fluid_density_kgm3": fluid_density + rng.normal(0.0, 0.4 * noise_scale, n),
        "efficiency": efficiency + rng.normal(0.0, 0.003 * noise_scale, n),
        "hydraulic_power_kw": hydraulic_power_kw + rng.normal(0.0, 0.04 * noise_scale, n),
        "shaft_power_kw": shaft_power_kw + rng.normal(0.0, 0.05 * noise_scale, n),
        "motor_power_kw": motor_power_kw + rng.normal(0.0, 0.05 * noise_scale, n),
        "motor_current_a": current_a + rng.normal(0.0, 0.08 * noise_scale, n),
        "vibration_mm_s": vibration_mm_s + rng.normal(0.0, 0.05 * noise_scale, n),
        "bearing_temp_c": bearing_temp_c + rng.normal(0.0, 0.18 * noise_scale, n),
        "npsha_m": npsha_m + rng.normal(0.0, 0.04 * noise_scale, n),
        "npshr_m": npshr_m + rng.normal(0.0, 0.03 * noise_scale, n),
        "cavitation_margin_m": cavitation_margin_m + rng.normal(0.0, 0.04 * noise_scale, n),
    })

    data["fault_cavitation"] = 0
    data["fault_impeller_wear"] = 0
    data["fault_bearing_friction"] = 0
    data["is_faulted"] = 0
    data["fault_labels"] = "healthy"
    data["fault_severity"] = "none"
    return data


In [10]:
def apply_cavitation_fault(df, severity, params):
    effective_npsha = df["npsha_m"].to_numpy() - (1.0 + 4.0 * severity)
    effective_margin = effective_npsha - df["npshr_m"].to_numpy()
    intensity = severity * np.clip((1.8 - effective_margin) / 2.2, 0.25, 1.0)

    df["flow_m3h"] *= 1.0 - 0.04 * intensity
    df["differential_head_m"] *= 1.0 - 0.10 * intensity
    df["efficiency"] *= 1.0 - 0.12 * intensity
    df["npsha_m"] = effective_npsha
    df["cavitation_margin_m"] = effective_margin

    df = recompute_power_columns(
        df=df,
        motor_efficiency=params["motor_efficiency"],
        line_voltage_v=params["line_voltage_v"],
        motor_power_factor=params["motor_power_factor"],
    )
    df["vibration_mm_s"] += 1.3 + 3.2 * intensity
    df["bearing_temp_c"] += 1.5 + 4.0 * intensity
    return df


def apply_impeller_wear_fault(df, severity, params):
    df["flow_m3h"] *= 1.0 - 0.03 * severity
    df["differential_head_m"] *= 1.0 - 0.12 * severity
    df["efficiency"] *= 1.0 - 0.08 * severity

    df = recompute_power_columns(
        df=df,
        motor_efficiency=params["motor_efficiency"],
        line_voltage_v=params["line_voltage_v"],
        motor_power_factor=params["motor_power_factor"],
    )
    df["vibration_mm_s"] += 0.4 + 0.9 * severity
    df["bearing_temp_c"] += 0.6 + 1.6 * severity
    return df


def apply_bearing_friction_fault(df, severity, params):
    loss_factor = 1.0 + 0.06 + 0.14 * severity
    df["shaft_power_kw"] *= loss_factor
    df["motor_power_kw"] = df["shaft_power_kw"] / params["motor_efficiency"]
    df["motor_current_a"] = motor_current_a(
        df["motor_power_kw"].to_numpy(),
        params["line_voltage_v"],
        params["motor_power_factor"],
    )
    df["vibration_mm_s"] += 0.7 + 1.8 * severity
    df["bearing_temp_c"] += 5.0 + 11.0 * severity
    return df

In [11]:
FAULT_REGISTRY = {
    "cavitation": apply_cavitation_fault,
    "impeller_wear": apply_impeller_wear_fault,
    "bearing_friction": apply_bearing_friction_fault,
}


def finalize_fault_labels(df, fault_type, severity):
    if fault_type == "healthy":
        return df

    df["active_fault"] = fault_type
    df[f"fault_{fault_type}"] = 1
    df["is_faulted"] = 1
    df["fault_labels"] = fault_type
    df["fault_severity"] = severity_label(severity)
    return df


def round_and_clip(df):
    df["speed_rpm"] = np.round(np.clip(df["speed_rpm"], 0.0, 1800.0), 1)
    df["flow_m3h"] = np.round(np.clip(df["flow_m3h"], 0.0, 240.0), 2)
    df["differential_head_m"] = np.round(np.clip(df["differential_head_m"], 0.0, 80.0), 2)
    df["suction_abs_pressure_bar"] = np.round(np.clip(df["suction_abs_pressure_bar"], 0.8, 3.0), 3)
    df["discharge_pressure_bar"] = np.round(np.clip(df["discharge_pressure_bar"], 0.8, 10.0), 3)
    df["ambient_temp_c"] = np.round(np.clip(df["ambient_temp_c"], 0.0, 50.0), 2)
    df["fluid_temp_c"] = np.round(np.clip(df["fluid_temp_c"], 0.0, 80.0), 2)
    df["fluid_density_kgm3"] = np.round(np.clip(df["fluid_density_kgm3"], 960.0, 1001.0), 2)
    df["efficiency"] = np.round(np.clip(df["efficiency"], 0.30, 0.90), 4)
    df["hydraulic_power_kw"] = np.round(np.clip(df["hydraulic_power_kw"], 0.0, 40.0), 3)
    df["shaft_power_kw"] = np.round(np.clip(df["shaft_power_kw"], 0.0, 55.0), 3)
    df["motor_power_kw"] = np.round(np.clip(df["motor_power_kw"], 0.0, 60.0), 3)
    df["motor_current_a"] = np.round(np.clip(df["motor_current_a"], 0.0, 120.0), 3)
    df["vibration_mm_s"] = np.round(np.clip(df["vibration_mm_s"], 0.0, 20.0), 3)
    df["bearing_temp_c"] = np.round(np.clip(df["bearing_temp_c"], 0.0, 120.0), 2)
    df["npsha_m"] = np.round(np.clip(df["npsha_m"], -10.0, 30.0), 3)
    df["npshr_m"] = np.round(np.clip(df["npshr_m"], 0.0, 15.0), 3)
    df["cavitation_margin_m"] = np.round(np.clip(df["cavitation_margin_m"], -15.0, 25.0), 3)
    return df

In [12]:
def simulate_device(device_config, seed=random_seed, sample_offset=0):
    rng = np.random.default_rng(seed)
    n_samples = int(device_config["n_samples"])
    params = make_device_params(rng, device_config.get("device_params"))

    fault_mix = dict(device_config.get("fault_mix", {"healthy": 1.0}))
    fault_types = list(fault_mix.keys())
    fault_probs = np.array(list(fault_mix.values()), dtype=float)
    fault_probs = fault_probs / fault_probs.sum()
    counts = rng.multinomial(n_samples, fault_probs)

    records = []
    current_offset = sample_offset

    for fault_type, count in zip(fault_types, counts):
        if count == 0:
            continue

        conditions = sample_conditions(rng, count, fault_type)
        severity = np.zeros(count, dtype=float) if fault_type == "healthy" else rng.uniform(0.15, 1.0, count)

        df = build_healthy_samples(
            device_id=device_config["device_id"],
            params=params,
            conditions=conditions,
            rng=rng,
            sample_offset=current_offset,
        )

        if fault_type != "healthy":
            df = FAULT_REGISTRY[fault_type](df, severity, params)
            df = finalize_fault_labels(df, fault_type, severity)

        records.append(df)
        current_offset += count

    data = pd.concat(records, ignore_index=True)
    data = data.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    data = round_and_clip(data)
    return data[ALL_COLUMNS], params

In [13]:
def simulate_station(device_configs, seed=random_seed):
    all_data = []
    device_rows = []
    sample_offset = 0

    for i, cfg in enumerate(device_configs):
        df, params = simulate_device(cfg, seed=seed + i + 1, sample_offset=sample_offset)
        all_data.append(df)
        sample_offset += len(df)

        row = {"device": cfg["device_id"]}
        row.update(params)
        device_rows.append(row)

    data = pd.concat(all_data, ignore_index=True)
    data = data.sort_values(["device", "sample_id"]).reset_index(drop=True)
    devices = pd.DataFrame(device_rows)
    return data, devices

In [14]:
device_configs = [
    {
        "device_id": "P001",
        "n_samples": 2500,
        "device_params": {"efficiency_peak": 0.83},
        "fault_mix": {
            "healthy": 0.70,
            "cavitation": 0.10,
            "impeller_wear": 0.10,
            "bearing_friction": 0.10,
        },
    },
    {
        "device_id": "P002",
        "n_samples": 2500,
        "device_params": {"q_bep_m3h": 88.0, "h_bep_m": 46.0},
        "fault_mix": {
            "healthy": 0.72,
            "cavitation": 0.08,
            "impeller_wear": 0.12,
            "bearing_friction": 0.08,
        },
    },
    {
        "device_id": "P003",
        "n_samples": 2500,
        "device_params": {"thermal_bias_c": 1.0},
        "fault_mix": {
            "healthy": 0.68,
            "cavitation": 0.12,
            "impeller_wear": 0.08,
            "bearing_friction": 0.12,
        },
    },
    {
        "device_id": "P004",
        "n_samples": 2500,
        "device_params": {"motor_efficiency": 0.92},
        "fault_mix": {
            "healthy": 0.75,
            "cavitation": 0.07,
            "impeller_wear": 0.08,
            "bearing_friction": 0.10,
        },
    },
]

In [15]:
df, devices = simulate_station(device_configs, seed=random_seed)

output_dir = Path("../generated_data/tabular")
output_dir.mkdir(parents=True, exist_ok=True)

df.to_parquet(output_dir / "centrifugal_pump_operating_data_v1.parquet", index=False)
devices.to_parquet(output_dir / "centrifugal_pump_device_metadata_v1.parquet", index=False)


In [16]:
df

,sample_id,device,active_fault,speed_rpm,flow_m3h,differential_head_m,suction_abs_pressure_bar,discharge_pressure_bar,ambient_temp_c,fluid_temp_c,...,bearing_temp_c,npsha_m,npshr_m,cavitation_margin_m,fault_cavitation,fault_impeller_wear,fault_bearing_friction,is_faulted,fault_labels,fault_severity
0,0,P001,healthy,1256.9,75.71,34.76,1.482,4.861,15.72,17.64,...,37.49,15.708,2.283,13.448,0,0,0,0,healthy,none
1,1,P001,healthy,1503.4,130.47,38.03,1.239,4.954,26.97,30.55,...,54.15,13.485,5.820,7.629,0,0,0,0,healthy,none
2,2,P001,healthy,1460.3,110.70,40.96,1.351,5.353,24.90,28.48,...,50.70,13.075,4.675,8.473,0,0,0,0,healthy,none
3,3,P001,healthy,1345.7,99.24,35.43,1.620,5.069,36.23,42.26,...,60.06,15.454,3.733,11.638,0,0,0,0,healthy,none
4,4,P001,healthy,1327.3,85.63,37.26,1.211,4.892,8.71,14.82,...,31.42,13.832,2.836,10.979,0,0,0,0,healthy,none
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9995,P004,bearing_friction,1457.6,129.52,31.45,1.343,4.386,15.33,21.37,...,52.45,15.514,4.326,11.294,0,0,1,1,bearing_friction,medium
9996,9996,P004,bearing_friction,1250.1,77.82,32.26,1.650,4.815,26.20,34.71,...,60.90,16.658,1.963,14.668,0,0,1,1,bearing_friction,high
9997,9997,P004,bearing_friction,1223.4,97.88,25.48,1.587,4.058,8.84,15.58,...,42.27,15.474,3.041,12.481,0,0,1,1,bearing_friction,medium
9998,9998,P004,bearing_friction,1315.5,96.01,31.74,1.455,4.553,23.79,29.78,...,53.90,14.802,3.051,11.824,0,0,1,1,bearing_friction,low
